In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

In [3]:
df = pd.read_csv("../data/nifty50_daily.csv")
df.head()

,Date,Open,High,Low,Close,Shares Traded,Turnover (₹ Cr)
0,2015-11-09,7788.25,7937.75,7771.70,7915.20,218422388.0,9376.17
1,2015-11-10,7877.60,7885.10,7772.85,7783.35,170267413.0,7153.47
2,2015-11-11,7838.80,7847.95,7819.10,7825.00,22380435.0,1123.44
3,2015-11-13,7762.45,7775.10,7730.90,7762.25,165876819.0,7731.55
4,2015-11-16,7732.95,7838.85,7714.15,7806.60,154134885.0,6871.15


In [4]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

df.tail()

,Date,Open,High,Low,Close,Shares Traded,Turnover (₹ Cr)
2143,2024-07-08,24329.45,24344.60,24240.55,24320.55,266299131.0,26356.03
2144,2024-07-09,24351.00,24443.60,24331.90,24433.20,250537091.0,29361.08
2145,2024-07-10,24459.85,24461.05,24141.80,24324.45,292263786.0,35358.54
2146,2024-07-11,24396.55,24402.65,24193.75,24315.95,306404194.0,32115.44
2147,2024-07-12,24387.95,24592.20,24331.15,24502.15,325823474.0,39565.33


In [5]:
features = [
    "Open",
    "High",
    "Low",
    "Close",
    "Shares Traded",
    "Turnover (₹ Cr)"
]

target = "Close"

X = df[features]
y = df[target]

In [6]:
train_end = "2021-12-31"
val_end = "2023-12-31"

X_train = X[df["Date"] <= train_end]
y_train = y[df["Date"] <= train_end]

X_val = X[(df["Date"] > train_end) & (df["Date"] <= val_end)]
y_val = y[(df["Date"] > train_end) & (df["Date"] <= val_end)]

X_test = X[df["Date"] > val_end]
y_test = y[df["Date"] > val_end]

len(X_train), len(X_val), len(X_test)

(1521, 494, 133)

In [7]:
# Shift close price by 1 day
y_pred_naive = y_test.shift(1)

# Drop first NaN
y_test_aligned = y_test.iloc[1:]
y_pred_naive = y_pred_naive.iloc[1:]

In [10]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

rmse_naive = np.sqrt(mean_squared_error(y_test_aligned, y_pred_naive))
mae_naive = mean_absolute_error(y_test_aligned, y_pred_naive)

rmse_naive, mae_naive

(np.float64(212.73026788635434), 136.2155303030305)

In [11]:
y_test_aligned.head(), y_pred_naive.head()

(2016    21665.80
 2017    21517.35
 2018    21658.60
 2019    21710.80
 2020    21513.00
 Name: Close, dtype: float64,
 2016    21741.90
 2017    21665.80
 2018    21517.35
 2019    21658.60
 2020    21710.80
 Name: Close, dtype: float64)

Linear Regression

In [12]:
# Number of lags
n_lags = 5

df_lr = df.copy()

for lag in range(1, n_lags + 1):
    df_lr[f"Close_lag_{lag}"] = df_lr["Close"].shift(lag)

df_lr.head(10)

,Date,Open,High,Low,Close,Shares Traded,Turnover (₹ Cr),Close_lag_1,Close_lag_2,Close_lag_3,Close_lag_4,Close_lag_5
0,2015-11-09,7788.25,7937.75,7771.70,7915.20,218422388.0,9376.17,NaN,NaN,NaN,NaN,NaN
1,2015-11-10,7877.60,7885.10,7772.85,7783.35,170267413.0,7153.47,7915.20,NaN,NaN,NaN,NaN
2,2015-11-11,7838.80,7847.95,7819.10,7825.00,22380435.0,1123.44,7783.35,7915.20,NaN,NaN,NaN
3,2015-11-13,7762.45,7775.10,7730.90,7762.25,165876819.0,7731.55,7825.00,7783.35,7915.20,NaN,NaN
4,2015-11-16,7732.95,7838.85,7714.15,7806.60,154134885.0,6871.15,7762.25,7825.00,7783.35,7915.20,NaN
5,2015-11-17,7848.75,7860.45,7793.00,7837.55,149451211.0,6367.14,7806.60,7762.25,7825.00,7783.35,7915.20
6,2015-11-18,7823.15,7843.40,7725.05,7731.80,148037721.0,6112.32,7837.55,7806.60,7762.25,7825.00,7783.35
7,2015-11-19,7788.50,7854.90,7765.45,7842.75,136702518.0,7410.15,7731.80,7837.55,7806.60,7762.25,7825.00
8,2015-11-20,7841.90,7906.95,7817.80,7856.55,156610433.0,7298.01,7842.75,7731.80,7837.55,7806.60,7762.25
9,2015-11-23,7869.50,7877.50,7825.20,7849.25,130871603.0,6099.09,7856.55,7842.75,7731.80,7837.55,7806.60


In [13]:
df_lr = df_lr.dropna().reset_index(drop=True)

In [14]:
feature_cols = [f"Close_lag_{lag}" for lag in range(1, n_lags + 1)]

X_lr = df_lr[feature_cols]
y_lr = df_lr["Close"]
dates_lr = df_lr["Date"]

In [15]:
X_train_lr = X_lr[dates_lr <= train_end]
y_train_lr = y_lr[dates_lr <= train_end]

X_val_lr = X_lr[(dates_lr > train_end) & (dates_lr <= val_end)]
y_val_lr = y_lr[(dates_lr > train_end) & (dates_lr <= val_end)]

X_test_lr = X_lr[dates_lr > val_end]
y_test_lr = y_lr[dates_lr > val_end]

In [16]:
from sklearn.linear_model import LinearRegression

lr_model = LinearRegression()
lr_model.fit(X_train_lr, y_train_lr)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [17]:
y_pred_lr = lr_model.predict(X_test_lr)

rmse_lr = np.sqrt(mean_squared_error(y_test_lr, y_pred_lr))
mae_lr = mean_absolute_error(y_test_lr, y_pred_lr)

rmse_lr, mae_lr

(np.float64(210.10812694031796), 132.17725767587152)

In [18]:
pd.Series(lr_model.coef_, index=feature_cols)

Close_lag_1    1.002248
Close_lag_2    0.005191
Close_lag_3   -0.035971
Close_lag_4    0.049727
Close_lag_5   -0.020867
dtype: float64